In [1]:
# =========================
# Stage 6 — Results packaging & analysis
# =========================
import os, joblib, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ---- Paths
BASE = r"C:\Users\nabal\Documents\FYP"
FEAT_DIR = os.path.join(BASE, "features_engineered_v1")
SPLIT_DIR = os.path.join(BASE, "splits")
MEL_DIR   = os.path.join(BASE, "mel_spectrograms_v1")
BL_DIR    = os.path.join(BASE, "models_baselines")
DEEP_DIR  = os.path.join(BASE, "models_deep")
REPORTS   = os.path.join(BASE, "reports"); os.makedirs(REPORTS, exist_ok=True)

# =========================
# 1) Classical (RF) on engineered features
# =========================
# Load engineered features
test_df  = pd.read_csv(os.path.join(FEAT_DIR, "test.csv"))
val_df   = pd.read_csv(os.path.join(FEAT_DIR, "val.csv"))
train_df = pd.read_csv(os.path.join(FEAT_DIR, "train.csv"))
NONFEATS = ["file_path","maqam","reciter"]
feat_cols = [c for c in test_df.columns if c not in NONFEATS]

X_te_eng, y_te_lab = test_df[feat_cols].values, test_df["maqam"].values
X_va_eng, y_va_lab = val_df[feat_cols].values,  val_df["maqam"].values

# Load RF model
rf_pkg = joblib.load(os.path.join(BL_DIR, "rf_engineered.joblib"))
rf = rf_pkg["pipeline"]; le_rf = rf_pkg["label_encoder"]

# Encode labels to ints for metrics (consistent)
if hasattr(le_rf, "transform"):
    y_te_i = le_rf.transform(y_te_lab)
    y_va_i = le_rf.transform(y_va_lab)
    classes = list(le_rf.classes_)
else:
    inv = {v:k for k,v in le_rf.items()}
    classes = [inv[i] for i in range(len(inv))]
    map_ = {c:i for i,c in enumerate(classes)}
    y_te_i = np.array([map_[c] for c in y_te_lab])
    y_va_i = np.array([map_[c] for c in y_va_lab])

# Re-evaluate RF
rf_te_pred = rf.predict(X_te_eng)
rf_va_pred = rf.predict(X_va_eng)
rf_te_acc  = accuracy_score(y_te_i, rf_te_pred)
rf_va_acc  = accuracy_score(y_va_i, rf_va_pred)
print(f"RF — Val {rf_va_acc:.3f} | Test {rf_te_acc:.3f}")

# Save RF confusion matrix (test)
cm_rf = confusion_matrix(y_te_i, rf_te_pred, normalize="true")
plt.figure(figsize=(4.8,4.2))
sns.heatmap(cm_rf, annot=True, cmap="Blues", xticklabels=classes, yticklabels=classes, fmt=".2f", cbar=False)
plt.title("RF (engineered) — Test Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
rf_cm_path = os.path.join(REPORTS, "rf_confusion_test.png")
plt.savefig(rf_cm_path, dpi=200); plt.close()
print("Saved:", rf_cm_path)

# =========================
# 2) CRNN on Mel spectrograms
# =========================
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load Mel arrays and label encoder (from Stage 5)
X_tr = np.load(os.path.join(MEL_DIR, "X_train.npy"))
y_tr = np.load(os.path.join(MEL_DIR, "y_train.npy"))
X_va = np.load(os.path.join(MEL_DIR, "X_val.npy"))
y_va = np.load(os.path.join(MEL_DIR, "y_val.npy"))
X_te = np.load(os.path.join(MEL_DIR, "X_test.npy"))
y_te = np.load(os.path.join(MEL_DIR, "y_test.npy"))
le_deep = joblib.load(os.path.join(MEL_DIR, "label_encoder.joblib"))
if hasattr(le_deep, "classes_"):
    classes_deep = list(le_deep.classes_)
else:
    inv = {v:k for k,v in le_deep.items()}
    classes_deep = [inv[i] for i in range(len(inv))]

assert classes == classes_deep, "Class order mismatch between RF & CRNN."

# Match test file paths for error analysis
test_split = pd.read_csv(os.path.join(SPLIT_DIR, "test.csv")).reset_index(drop=True)  # has file_path, maqam, reciter

# CRNN definition (must match Stage 5)
class CRNN(nn.Module):
    def __init__(self, n_classes=3, n_mels=128, cnn_out=128, rnn_hidden=128, rnn_layers=1, bidir=True, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(64, cnn_out, kernel_size=3, padding=1), nn.BatchNorm2d(cnn_out), nn.ReLU(),
        )
        self.gru = nn.GRU(input_size=cnn_out*(n_mels//4), hidden_size=128,
                          num_layers=1, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(128*2, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        z = self.features(x)          # (B, C, F/4, T/4)
        B,C,F,T = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B, T, C*F)
        out,_ = self.gru(z)           # (B, T, 256)
        out = out.mean(dim=1)         # (B, 256)
        return self.classifier(out)   # (B, n_classes)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
crnn = CRNN(n_classes=len(classes)).to(DEVICE)

ckpt = torch.load(os.path.join(DEEP_DIR, "crnn_mel_best.pth"), map_location=DEVICE)
crnn.load_state_dict(ckpt["state_dict"]); crnn.eval()

# Build a small DataLoader for test
class MelSet(Dataset):
    def __init__(self, X, y): self.X=X.astype(np.float32); self.y=y.astype(np.int64)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i][None,:,:]), torch.tensor(self.y[i])

test_loader = DataLoader(MelSet(X_te, y_te), batch_size=16, shuffle=False)

# Predict
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = crnn(xb)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        y_prob.append(probs)
        y_pred.extend(probs.argmax(1).tolist())
        y_true.extend(yb.numpy().tolist())
y_prob = np.vstack(y_prob)

crnn_te_acc = accuracy_score(y_true, y_pred)
print(f"CRNN — Test {crnn_te_acc:.3f}")

# Save CRNN confusion matrix (test)
cm_crnn = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(4.8,4.2))
sns.heatmap(cm_crnn, annot=True, cmap="Purples", xticklabels=classes, yticklabels=classes, fmt=".2f", cbar=False)
plt.title("CRNN (Mel) — Test Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
crnn_cm_path = os.path.join(REPORTS, "crnn_confusion_test.png")
plt.savefig(crnn_cm_path, dpi=200); plt.close()
print("Saved:", crnn_cm_path)

# =========================
# 3) Misclassification report (CRNN)
# =========================
mis_idx = [i for i,(t,p) in enumerate(zip(y_true,y_pred)) if t!=p]
mis_rows = []
for i in mis_idx:
    row = {
        "file_path": test_split.loc[i,"file_path"],
        "true": classes[y_true[i]],
        "pred": classes[y_pred[i]],
        "conf_pred": float(y_prob[i, y_pred[i]]),
        # top-3 for insight
        "top3_labels": [classes[j] for j in np.argsort(-y_prob[i])[:3].tolist()],
        "top3_probs":  [float(x) for x in np.sort(y_prob[i])[::-1][:3].tolist()]
    }
    mis_rows.append(row)

mis_df = pd.DataFrame(mis_rows)
mis_csv = os.path.join(REPORTS, "crnn_misclassified_test.csv")
mis_df.to_csv(mis_csv, index=False)
print("Saved misclassifications:", mis_csv)

# =========================
# 4) Tiny robustness probe (SpecAugment at inference)
# =========================
def time_mask_np(spec, max_w=30):
    S = spec.copy(); T = S.shape[1]
    w = np.random.randint(0, max_w+1)
    if 0 < w < T:
        t0 = np.random.randint(0, T-w+1)
        S[:, t0:t0+w] = 0.0
    return S

def freq_mask_np(spec, max_w=16):
    S = spec.copy(); F = S.shape[0]
    w = np.random.randint(0, max_w+1)
    if 0 < w < F:
        f0 = np.random.randint(0, F-w+1)
        S[f0:f0+w, :] = 0.0
    return S

X_te_aug = []
for i in range(len(X_te)):
    S = X_te[i]
    if np.random.rand() < 0.7: S = time_mask_np(S, max_w=36)
    if np.random.rand() < 0.7: S = freq_mask_np(S, max_w=18)
    X_te_aug.append(S)
X_te_aug = np.stack(X_te_aug).astype(np.float32)

aug_loader = DataLoader(MelSet(X_te_aug, y_te), batch_size=16, shuffle=False)
y_pred_aug = []
with torch.no_grad():
    for xb, yb in aug_loader:
        xb = xb.to(DEVICE)
        y_pred_aug.extend(crnn(xb).argmax(1).cpu().numpy().tolist())
acc_aug = accuracy_score(y_true, y_pred_aug)

# =========================
# 5) Save summary table + pretty text report
# =========================
summary = pd.DataFrame([
    {"Model":"RF (engineered)", "Val Acc": rf_va_acc, "Test Acc": rf_te_acc},
    {"Model":"CRNN (Mel)",      "Val Acc": None,      "Test Acc": crnn_te_acc, "Test Acc (SpecAug)": acc_aug}
])
sum_csv = os.path.join(REPORTS, "stage6_summary.csv")
summary.to_csv(sum_csv, index=False)
print("Saved summary:", sum_csv)
print(summary)

# Text report
text = {
  "classes": classes,
  "rf": {"val_acc": float(rf_va_acc), "test_acc": float(rf_te_acc),
         "confusion_png": os.path.basename(rf_cm_path)},
  "crnn": {"test_acc": float(crnn_te_acc), "confusion_png": os.path.basename(crnn_cm_path),
           "robustness_specaug_test_acc": float(acc_aug)},
  "artifacts": {
      "rf_model": "models_baselines/rf_engineered.joblib",
      "crnn_ckpt": "models_deep/crnn_mel_best.pth",
      "crnn_onnx": "models_deep/crnn_mel_best.onnx",
      "misclassified_csv": "reports/crnn_misclassified_test.csv",
      "summary_csv": "reports/stage6_summary.csv"
  }
}
with open(os.path.join(REPORTS, "stage6_report.json"), "w", encoding="utf-8") as f:
    json.dump(text, f, indent=2)
print("📝 Saved JSON report →", os.path.join(REPORTS, "stage6_report.json"))


RF — Val 0.920 | Test 0.923
Saved: C:\Users\nabal\Documents\FYP\reports\rf_confusion_test.png
CRNN — Test 0.962
Saved: C:\Users\nabal\Documents\FYP\reports\crnn_confusion_test.png
Saved misclassifications: C:\Users\nabal\Documents\FYP\reports\crnn_misclassified_test.csv
Saved summary: C:\Users\nabal\Documents\FYP\reports\stage6_summary.csv
             Model  Val Acc  Test Acc  Test Acc (SpecAug)
0  RF (engineered)     0.92  0.923077                 NaN
1       CRNN (Mel)      NaN  0.961538            0.884615
📝 Saved JSON report → C:\Users\nabal\Documents\FYP\reports\stage6_report.json
